# Latency budgets, parallelism & speculation: engineer latency on purpose

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 40 §40.5 · type: concept-lab*

**The promise:** by the end you can decompose a request's latency into a per-stage budget, cut the biggest line with bounded parallelism, judge when speculation pays, and gate cost/latency regressions in CI.

## 🧠 Why this matters

Cost is paid by the company; **latency is paid by the user**, on every single interaction. The discipline is the one you applied to web services in Ch 24 — set a budget, decompose it, attack the biggest stage — with two LLM-specific twists:

- **TTFT, not total**, is the metric for chat. A response that *starts* in 400 ms feels fast even at 8 s total; the same answer delivered all at once after 8 s feels broken.
- **Output tokens dominate** generation time. Models decode sequentially, so a 2,000-token answer takes ~10× a 200-token one. The fastest latency fix is often *ask for less*.

Everything here runs on a **deterministic, seeded timing simulator** — no network, no real latency, fully reproducible. See §40.5.

## Objectives & prereqs

**By the end you can:**
- Separate **TTFT** from total time and explain why streaming changes *perceived* latency.
- Model generation time as ∝ output tokens and predict the speedup from halving `max_tokens`.
- Reproduce the book's **per-stage latency budget** table and give each stage an owner.
- Run the book's bounded `summarize_all` (asyncio.gather + a Semaphore) over a mock async client.
- Model the three **speculative patterns** and judge when wasted tokens buy wall-clock.
- Wire a **cost+latency budget assertion** that fails offline when a change blows the budget.

**Prereqs:** `40-01`, `40-02`; Ch 4 (async) and Ch 24 (latency budgets) read.

**Runs free & offline.** Mock async client with *simulated* latencies; deterministic and seeded. No API key, no real waiting.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import time
import random
import asyncio

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default): a mock async client with simulated (not real) latencies. The
# parallelism structure is identical against a live AsyncAnthropic client.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(40)  # the timing model is seeded so every number below is reproducible

print("MOCK mode:", MOCK, "— offline, deterministic" if MOCK else "— LIVE (would call AsyncAnthropic)")

## TTFT vs. total — why streaming *feels* fast

Streaming doesn't change total wall-clock; it changes *when the first token lands*. We model one call as a fixed TTFT followed by steady decode, and compare what a buffered vs. a streamed user experiences.

In [ ]:
TTFT_S = 0.40          # seconds before the first token (network + queue + prefill)
DECODE_RATE = 50.0     # tokens / second
OUTPUT_TOKENS = 400

total = TTFT_S + OUTPUT_TOKENS / DECODE_RATE
print(f"total wall-clock (both modes)   : {total:.2f}s")
print(f"first output, BUFFERED          : {total:.2f}s  (spinner the whole time)")
print(f"first output, STREAMED          : {TTFT_S:.2f}s  (user reads along)")
print("\nSame total. Streaming transforms PERCEIVED latency — so budget TTFT separately.")

## 🔮 Predict: halve `max_tokens`

Generation time is roughly proportional to output tokens (decode is sequential). The current call generates 400 output tokens.

**Predict, before running:** if you halve `max_tokens` so the model emits ~200 tokens instead of 400, what happens to *total* latency — does it halve, drop by a third, or barely move? Remember TTFT is fixed. Write your guess, then measure.

In [ ]:
def gen_latency(output_tokens, ttft=TTFT_S, decode_rate=DECODE_RATE):
    """time ~= TTFT + output_tokens / decode_rate. Output dominates the tail."""
    return ttft + output_tokens / decode_rate


full = gen_latency(400)
half = gen_latency(200)
print(f"max_tokens ~400 -> {full:.2f}s")
print(f"max_tokens ~200 -> {half:.2f}s")
print(f"speedup: {full / half:.2f}x  (NOT 2x — the fixed TTFT floor dilutes the gain)")
print("\n'Ask for less' is latency engineering — but TTFT sets a floor you can't decode past.")

**What you just saw.** Halving output does *not* halve latency, because TTFT is a fixed floor that decode time is added on top of. Output discipline is real latency engineering — but only the decode portion shrinks, so the win is biggest on long answers and smallest on short ones.

## The per-stage latency budget

A budget per stage means a regression has an **owner** — not a vague "it feels slow." This reproduces the book's table for a streaming RAG chat turn. The numbers are illustrative; the discipline is having a line and a lever per stage.

In [ ]:
LATENCY_BUDGET = [
    # stage,            budget_ms, owner,         levers
    ("network + auth",        50,  "platform",    "keep-alive connections, regional endpoints"),
    ("retrieval",            150,  "search",      "index tuning, caching, smaller candidate sets (Ch 13)"),
    ("TTFT",                 600,  "llm-gateway", "prompt caching, shorter prompts, faster tier"),
    ("generation",          3000,  "llm-gateway", "shorter outputs, streaming so the user reads along"),
    ("tool calls",           400,  "agent",       "parallelize independent calls, cache tool results"),
]

print(f"{'stage':16s} {'budget':>8s}  {'owner':12s} levers")
total_budget = 0
for stage, ms, owner, levers in LATENCY_BUDGET:
    total_budget += ms
    print(f"{stage:16s} {ms:>6d}ms  {owner:12s} {levers}")
print(f"{'TOTAL':16s} {total_budget:>6d}ms")
print("\nA regression in any line now has exactly one owner to page.")

## Parallelism — the book's bounded `summarize_all`

Agents are full of sequential habits that don't need to be sequential. Fan-out over ten documents shouldn't be a loop — run it with **bounded** concurrency. This is the book's `summarize_all`: `asyncio.gather` plus a `Semaphore` to respect rate limits, over a **mock** async client with simulated per-call latency.

In [ ]:
class MockAsyncClient:
    """Stands in for anthropic.AsyncAnthropic. Each call 'takes' a simulated model
    time; we sleep a scaled, tiny real fraction so the notebook stays fast and offline."""

    SIM_LATENCY_S = 0.5  # each call 'costs' ~0.5s of model time

    async def summarize(self, doc):
        await asyncio.sleep(self.SIM_LATENCY_S * 0.02)  # scaled down so the cell runs fast
        return f"summary({doc})"


aclient = MockAsyncClient()


async def summarize_all(docs, limit=8):
    sem = asyncio.Semaphore(limit)  # respect rate limits (Ch 29 backpressure)

    async def bounded(doc):
        async with sem:
            return await aclient.summarize(doc)

    return await asyncio.gather(*(bounded(d) for d in docs))


docs = [f"doc-{i}" for i in range(10)]

start = time.monotonic()
results = await summarize_all(docs, limit=8)
elapsed = time.monotonic() - start
# Report in 'model-seconds' (un-scale the 0.02 factor) so the speedup is legible.
print(f"summarized {len(results)} docs at concurrency 8 in ~{elapsed / 0.02:.1f} model-seconds")
print("Ten docs at concurrency eight finish in ~the time of two — not ten.")

## ⚠️ Pitfall: unbounded `gather` meets the rate limiter

Drop the semaphore and `gather` fires *all* requests at once. For ten docs that's fine; for a thousand it slams straight into the provider's rate limiter — `429`s, retries, and a slower result than the bounded version (Ch 29's backpressure lesson, replayed). The semaphore is not optional at scale; it is the thing standing between you and your own rate limit.

In [ ]:
RATE_LIMIT = 8  # provider allows 8 concurrent in-flight requests


def simulate_inflight(n_requests, concurrency):
    """How many requests would exceed the provider's concurrency limit at peak?"""
    peak = min(n_requests, concurrency)
    rejected = max(0, peak - RATE_LIMIT)
    return peak, rejected


for conc in (8, 100):
    peak, rejected = simulate_inflight(1000, conc)
    label = "bounded (Semaphore=8)" if conc == 8 else "UNBOUNDED gather"
    verdict = "fine" if rejected == 0 else "429s + retries (slower!)"
    print(f"{label:22s}: peak in-flight {peak:>4d}, over rate limit by {rejected:>4d} -> {verdict}")

## Speculative patterns — spend tokens to buy time

When parallelism *within* a task is exhausted, spend a little cost to buy latency. Three patterns recur, and all three trade wasted tokens on the losing branch for wall-clock on the winning one — a good trade exactly when latency outvalues tokens:

- **Race the tiers** — fire cheap + frontier together; serve the cheap answer if a quick check accepts it, else the frontier result already in flight.
- **Prefetch** — while the user reads, speculatively run the retrieval (or draft) for the likely follow-up.
- **Optimistic tool calls** — start the lookup the plan almost certainly needs while the model is still reasoning.

> ⚠️ Don't confuse these app-level patterns with Ch 39's in-decoder *speculative decoding* — a different layer entirely.

In [ ]:
def race_the_tiers(cheap_ms, frontier_ms, cheap_accept_prob, cheap_tokens, frontier_tokens):
    """Model 'race the tiers': both run; wall-clock is the one we serve, cost is both."""
    # We serve the cheap answer when it's accepted (arrives first); else the frontier.
    expected_latency = (cheap_accept_prob * cheap_ms
                        + (1 - cheap_accept_prob) * frontier_ms)
    wasted_tokens = (cheap_accept_prob * frontier_tokens       # frontier wasted when cheap wins
                     + (1 - cheap_accept_prob) * cheap_tokens)  # cheap wasted when it loses
    return expected_latency, wasted_tokens


seq_latency = 300 + 1500   # cheap-then-fallback-to-frontier, in series
race_latency, wasted = race_the_tiers(
    cheap_ms=300, frontier_ms=1500, cheap_accept_prob=0.7,
    cheap_tokens=60, frontier_tokens=200,
)
print(f"sequential (cheap, then frontier on reject): {seq_latency:.0f}ms worst case")
print(f"race the tiers (both in flight)           : {race_latency:.0f}ms expected")
print(f"  cost of the race: ~{wasted:.0f} wasted tokens on the losing branch")
print("\nGood trade when a user-facing ms is worth more than a token. Usually it is.")

**What you just saw.** Racing the tiers spends tokens on a branch you throw away, but it collapses the worst-case latency — you never pay the full frontier wait *plus* a failed cheap attempt in series. Speculation is the deliberate purchase of wall-clock with tokens; it pays exactly when latency outvalues tokens, which for user-facing products it usually does.

## Regression guard — fail the build when the budget blows

The CI hook the book wants: assert this run stays under N tokens / T ms, so a prompt change that doubles tokens **fails offline** before it ships. Cost and latency budgets belong in CI alongside your tests (feeds `templates/github-actions-ci/`).

In [ ]:
def assert_within_budget(*, tokens, latency_ms, max_tokens, max_latency_ms):
    """Raise if a run blows its cost or latency budget. This is your CI gate."""
    problems = []
    if tokens > max_tokens:
        problems.append(f"tokens {tokens} > budget {max_tokens}")
    if latency_ms > max_latency_ms:
        problems.append(f"latency {latency_ms:.0f}ms > budget {max_latency_ms}ms")
    if problems:
        raise AssertionError("BUDGET REGRESSION: " + "; ".join(problems))
    return True


# A 'good' run passes.
ok = assert_within_budget(tokens=1800, latency_ms=2200, max_tokens=4000, max_latency_ms=4200)
print("baseline run within budget:", ok)

# Simulate a prompt change that doubled the tokens — the guard catches it offline.
try:
    assert_within_budget(tokens=4200, latency_ms=2200, max_tokens=4000, max_latency_ms=4200)
except AssertionError as e:
    print("CI would FAIL ->", e)

## 🎯 Senior lens

Cost and latency couple through **one** decision: *how much model do you buy per step?* Routing a step to a smaller model improves **both** at once — fewer parameters and shorter outputs mean less compute per token and fewer tokens — which makes model routing the single highest-leverage knob in this chapter, and the gateway (Ch 39) the place where it turns. Profile a slow agent like any distributed system: find the critical path first (Ch 23 traces), *then* tune. The number of **sequential** model calls usually dominates everything else, so the senior move is structural — combine steps, parallelize independent branches, route trivial decisions to fast models or plain code — long before reaching for parameter tweaks.

## 📋 The §40 cost/latency production checklist

A copyable, end-of-chapter checklist. Paste it into a PR template or an ADR — every item maps to something you built across 40-01 / 40-02 / 40-03.

In [ ]:
CH40_CHECKLIST = "\n".join([
    "Cost / Latency production checklist (book §40):",
    "",
    "[ ] Metering          - every call records tokens, cost, latency, model, feature,",
    "                        tenant at the GATEWAY, not scattered through app code.   (40-01)",
    "[ ] Unit economics    - cost per completed TASK (not per request) on a weekly dash. (40-01)",
    "[ ] Tails             - p95/p99 cost & latency tracked and alerted, not just means. (40-01)",
    "[ ] Caps              - hard max tokens, agent iterations, per-tenant spend.       (40-01)",
    "[ ] Model routing     - each step on the cheapest model that passes evals (Ch 22).",
    "[ ] Prompt cache      - stable prefix first, volatile last, cache_read verified >0. (40-02)",
    "[ ] App caches        - exact cache for repeats; semantic only where near-miss is",
    "                        OK, threshold tested, keys scoped per tenant.            (40-02)",
    "[ ] Batching          - latency-tolerant work on the discounted batch endpoint.",
    "[ ] Streaming         - user-facing responses streamed, TTFT budgeted separately. (40-03)",
    "[ ] Parallelism       - independent calls / fan-out concurrent, bounded by a Semaphore. (40-03)",
    "[ ] Output discipline - concise-output prompts, max_tokens set deliberately.      (40-03)",
    "[ ] Context compaction- long runs summarize history so per-turn tokens stay bounded (Ch 16).",
    "[ ] Regression guard  - cost & latency budgets in CI so a token-doubling change fails. (40-03)",
])
print(CH40_CHECKLIST)

## Recap

- **TTFT, not total**, is the chat metric; streaming changes *perceived* latency, not the wall-clock — budget it separately.
- **Output tokens dominate** generation time, but TTFT is a fixed floor — halving output does *not* halve latency.
- A **per-stage budget** gives every regression an owner; reproduce the book's table and assign levers.
- **Bounded** `summarize_all` (gather + Semaphore) parallelizes fan-out safely; unbounded gather meets the rate limiter.
- **Speculation** (race the tiers / prefetch / optimistic tool calls) spends wasted tokens to buy wall-clock — worth it when latency outvalues tokens.
- A **regression guard** asserting tokens/latency budgets in CI catches a token-doubling change *offline*.
- Cost and latency couple through one knob — **how much model per step** — and the gateway is where it turns.

## Exercises

Each one *changes* something and asks you to predict first. (Solutions arrive in `solutions/` in Phase 2.)

1. **Find the streaming win.** For what `OUTPUT_TOKENS` does halving `max_tokens` save at least 1 second of *total* latency? Predict above/below 100 tokens, then solve with `gen_latency`.
2. **Tune concurrency.** Re-run `summarize_all` over 40 docs at limits 1, 4, 16. Predict the ordering of wall-clock times and whether limit 16 beats limit 8 here, then measure.
3. **When does racing pay?** In `race_the_tiers`, find the `cheap_accept_prob` at which expected latency equals the frontier-only latency (1500ms). Predict it first.
4. **Tighten the guard.** Set `max_latency_ms=2000` and re-run the baseline through `assert_within_budget`. Predict pass or fail, then explain which stage of the budget table you'd attack to make it pass.

In [ ]:
# Exercise 1 — your code here.

In [ ]:
# Exercise 2 — your code here.

In [ ]:
# Exercise 3 — your code here.

In [ ]:
# Exercise 4 — your code here.

## Next

- **End of Ch 40's labs.** Next chapter: [`../41-security-safety-compliance/`](../41-security-safety-compliance/) — spend caps return as a *denial-of-wallet* abuse control on the same meter.
- **Book:** §40.5 (latency budgets, parallelism, speculation), the cost↔latency key idea, and the §40 checklist; Ch 4 (async), Ch 24 (web-service latency budgets), Ch 39 (the routing knob).
- **Blueprint this feeds:** [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) (the routing knob, streaming, bounded fan-out) and the CI budget check in [`templates/github-actions-ci/`](../../../templates/github-actions-ci/).
- **Capstone:** advances [`capstone/llm/gateway.py`](../../../capstone/llm/gateway.py) — the metering/budget hooks, caching, and prefix-aware routing the cost dashboard reads (`checkpoints/ch40-cost-and-caching`).